In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42

In [3]:
CSV_PATH = "loan_approval_decision_tree.csv"

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (600, 8)


,age,monthly_income,credit_score,employment_years,loan_amount,existing_loans,previous_default,loan_approved
0,25,105512,766,8,1440527,2,No,1
1,55,128618,639,9,816553,1,Yes,0
2,50,122944,691,9,980964,1,No,1
3,40,37106,455,0,922896,1,No,0
4,40,135300,562,11,1498089,5,Yes,0


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   age               600 non-null    int64 
 1   monthly_income    600 non-null    int64 
 2   credit_score      600 non-null    int64 
 3   employment_years  600 non-null    int64 
 4   loan_amount       600 non-null    int64 
 5   existing_loans    600 non-null    int64 
 6   previous_default  600 non-null    object
 7   loan_approved     600 non-null    int64 
dtypes: int64(7), object(1)
memory usage: 37.6+ KB


In [10]:
display(df.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,600.0,NaN,NaN,NaN,43.023333,13.055658,21.0,32.0,43.0,55.0,65.0
monthly_income,600.0,NaN,NaN,NaN,90140.436667,40560.721165,18621.0,55578.75,91898.5,123655.0,159873.0
credit_score,600.0,NaN,NaN,NaN,656.768333,84.579612,350.0,602.75,660.0,712.0,850.0
employment_years,600.0,NaN,NaN,NaN,6.91,4.805245,0.0,3.0,7.0,10.0,22.0
loan_amount,600.0,NaN,NaN,NaN,755170.801667,418435.696639,50823.0,393658.0,747879.5,1102706.75,1498302.0
existing_loans,600.0,NaN,NaN,NaN,2.42,1.73646,0.0,1.0,2.0,4.0,5.0
previous_default,600,2,No,493,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_approved,600.0,NaN,NaN,NaN,0.471667,0.499613,0.0,0.0,0.0,1.0,1.0


In [11]:
print("Missing values per column:")
display(df.isna().sum())

print("\nTarget class counts:")
display(df["loan_approved"].value_counts().rename(index={0: "Not approved", 1: "Approved"}))

print("\nTarget class proportions:")
display(df["loan_approved"].value_counts(normalize=True).rename(index={0: "Not approved", 1: "Approved"}).round(3))

Missing values per column:


age                 0
monthly_income      0
credit_score        0
employment_years    0
loan_amount         0
existing_loans      0
previous_default    0
loan_approved       0
dtype: int64


Target class counts:


loan_approved
Not approved    317
Approved        283
Name: count, dtype: int64


Target class proportions:


loan_approved
Not approved    0.528
Approved        0.472
Name: proportion, dtype: float64

In [12]:
X = df.drop(columns="loan_approved")
y = df["loan_approved"]

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())


X shape: (600, 7)
y shape: (600,)


,age,monthly_income,credit_score,employment_years,loan_amount,existing_loans,previous_default
0,25,105512,766,8,1440527,2,No
1,55,128618,639,9,816553,1,Yes
2,50,122944,691,9,980964,1,No
3,40,37106,455,0,922896,1,No
4,40,135300,562,11,1498089,5,Yes


In [13]:
categorical_features = ["previous_default"]
numeric_features = [col for col in X.columns if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features)
    ],
    verbose_feature_names_out=False
)